# Notebook 03 — Mô phỏng Bài nộp Sinh viên
**Nhóm 67 | Tuần 2**

Mô phỏng 200 bài nộp có lỗi cố ý để đo FPR thực tế (RQ1).

> Chạy notebook 00 trước.

In [ ]:
# [LOCAL] Bỏ Google Drive mount
# [LOCAL] Bỏ drive.mount


Mounted at /content/drive


In [1]:
import sys, json, csv
from pathlib import Path

BASE = Path("..")
sys.path.insert(0, str(BASE / "src"))
from runner_v2 import grade_submission, compute_fpr

OUT_DIR = BASE / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset
with open(BASE / "data" / "processed" / "hidden_v2.json", encoding="utf-8") as f:
    problems = {p["task_id"]: p for p in json.load(f)}

print(f"✓ Đọc được {len(problems)} bài")

✓ Đọc được 50 bài


In [2]:
# 100 bài nộp mô phỏng từ simulated_submissions.json
with open(BASE / "data" / "processed" / "submissions_50.json", encoding="utf-8") as f:
    submissions_raw = json.load(f)

submissions = []
for sub in submissions_raw:
    submissions.append({
        'task_id': sub['task_id'],
        'sv_id': sub['submission_id'],
        'mo_ta_loi': f"Lỗi {sub['error_type']} - {sub['note']}",
        'code': sub['submitted_code']
    })


In [3]:
print("=" * 70)
print(f"  MÔ PHỎNG {len(submissions)} BÀI NỘP SINH VIÊN")
print("=" * 70)

sim_rows = []
fp_count = 0

for sub in submissions:
    tid   = sub["task_id"]
    sv_id = sub["sv_id"]
    code  = sub["code"]
    prob  = problems.get(tid)
    if not prob:
        print(f"  [!] task_id {tid} không tìm thấy trong dataset")
        continue

    func = [
        l.split("(")[0].replace("def ", "").strip()
        for l in code.split("\n")
        if l.strip().startswith("def ")
    ]
    if not func:
        print(f"  [{sv_id}] Không tìm được tên hàm")
        continue
    func_name = func[0]

    pub_r = grade_submission(code, func_name, prob["public_tests"], "public")
    hid_r = grade_submission(code, func_name, prob["hidden_tests"], "hidden")
    fpr   = compute_fpr(pub_r, hid_r)

    if fpr["is_false_positive"]:
        fp_count += 1
        fp_flag = " ★ FALSE POSITIVE"
    else:
        fp_flag = ""

    print(f"[{sv_id}] Bài {tid} — {func_name}()")
    print(f"  Lỗi : {sub['mo_ta_loi']}")
    print(f"  Public : {pub_r['pass_count']}/{pub_r['total_count']} ({pub_r['test_pass_rate']}%)")
    print(f"  Hidden : {hid_r['pass_count']}/{hid_r['total_count']} ({hid_r['test_pass_rate']}%){fp_flag}")
    print()

    sim_rows.append({
        "sv_id":             sv_id,
        "task_id":           tid,
        "func":              func_name,
        "mo_ta_loi":         sub["mo_ta_loi"],
        "pub_pass":          pub_r["pass_count"],
        "pub_total":         pub_r["total_count"],
        "pub_tpr":           pub_r["test_pass_rate"],
        "pub_errors":        str(pub_r["error_counts"]),
        "hid_pass":          hid_r["pass_count"],
        "hid_total":         hid_r["total_count"],
        "hid_tpr":           hid_r["test_pass_rate"],
        "hid_errors":        str(hid_r["error_counts"]),
        "is_false_positive": fpr["is_false_positive"],
    })

# Kết luận RQ1
total   = len(sim_rows)
fpr_pct = round(fp_count / total * 100, 2)

print("=" * 55)
print("  KẾT LUẬN RQ1")
print("=" * 55)
print(f"  Tổng bài nộp kiểm tra : {total}")
print(f"  False Positive (FP)    : {fp_count}/{total} bài")
print(f"  False Positive Rate    : {fpr_pct}%")
print()
print("  → Với 3 test/bài, hệ thống bỏ sót lỗi edge case.")
print("  → Thêm hidden test giúp phát hiện các lỗi này.")
print("=" * 55)

# Lưu
sim_json = {"tong_submissions": total, "fp_count": fp_count,
            "fpr_rate_%": fpr_pct, "submissions": sim_rows}

with open(OUT_DIR / "student_simulation.json", "w", encoding="utf-8") as f:
    json.dump(sim_json, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "student_simulation.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(sim_rows[0].keys()))
    w.writeheader()
    w.writerows(sim_rows)

print(f"\n✓ Lưu: {OUT_DIR / 'student_simulation.json'}")
print(f"✓ Lưu: {OUT_DIR / 'student_simulation.csv'}")

  MÔ PHỎNG 100 BÀI NỘP SINH VIÊN
[SV001] Bài 1 — sum_list()
  Lỗi : Lỗi SE - Code bi loi cu phap (Syntax Error) truoc khi chay.
  Public : 0/3 (0.0%)
  Hidden : 0/10 (0.0%)

[SV002] Bài 1 — sum_list()
  Lỗi : Lỗi WA - Code chay duoc nhung tra ve ket qua sai (Wrong Answer) vi logic loi.
  Public : 1/3 (33.33%)
  Hidden : 3/10 (30.0%)

[SV003] Bài 2 — is_even()
  Lỗi : Lỗi SE - Code bi loi cu phap (Syntax Error) truoc khi chay.
  Public : 0/3 (0.0%)
  Hidden : 0/10 (0.0%)

[SV004] Bài 2 — is_even()
  Lỗi : Lỗi WA - Code chay duoc nhung tra ve ket qua sai (Wrong Answer) vi logic loi.
  Public : 1/3 (33.33%)
  Hidden : 1/10 (10.0%)

[SV005] Bài 3 — reverse_string()
  Lỗi : Lỗi SE - Code bi loi cu phap (Syntax Error) truoc khi chay.
  Public : 0/3 (0.0%)
  Hidden : 0/10 (0.0%)

[SV006] Bài 3 — reverse_string()
  Lỗi : Lỗi WA - Code chay duoc nhung tra ve ket qua sai (Wrong Answer) vi logic loi.
  Public : 3/3 (100.0%)
  Hidden : 10/10 (100.0%)

[SV007] Bài 4 — find_max()
  Lỗi : Lỗi SE - Co